In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2703531 entries, 0 to 2703530
Data columns (total 8 columns):
 #   Column     Dtype  
---  ------     -----  
 0   Ticker     object 
 1   Date       object 
 2   Open       float64
 3   High       float64
 4   Low        float64
 5   Close      float64
 6   Adj Close  float64
 7   Volume     int64  
dtypes: float64(5), int64(1), object(2)
memory usage: 165.0+ MB


In [8]:
import joblib

joblib.dump(model, "../models/stock_direction_model.pkl")
joblib.dump(features, "../models/features.pkl")

['../models/features.pkl']

In [7]:
latest_data = stock[features].iloc[[-1]]

prediction = model.predict(latest_data)[0]
confidence = model.predict_proba(latest_data)[0].max()

result = "Up" if prediction == 1 else "Down"

print("Prediction:", result)
print("Confidence:", round(confidence * 100, 2), "%")

Prediction: Down
Confidence: 83.0 %


In [6]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

features = [
    "Open",
    "High",
    "Low",
    "Close",
    "Volume",
    "Daily_Return",
    "MA_5",
    "MA_10",
    "Volume_Change",
    "Volatility_5"
]

X = stock[features]
y = stock["Target"]

split = int(len(stock) * 0.8)

X_train = X.iloc[:split]
X_test = X.iloc[split:]
y_train = y.iloc[:split]
y_test = y.iloc[split:]

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

preds = model.predict(X_test)

accuracy = accuracy_score(y_test, preds)
print("Accuracy:", accuracy)
print(classification_report(y_test, preds))

Accuracy: 0.4744859101294745
              precision    recall  f1-score   support

           0       0.47      0.99      0.64       619
           1       0.64      0.01      0.03       694

    accuracy                           0.47      1313
   macro avg       0.56      0.50      0.33      1313
weighted avg       0.56      0.47      0.32      1313



In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/SP500_Historical_Data.csv")
df.head()


,Ticker,Date,Open,High,Low,Close,Adj Close,Volume
0,A,2000-01-03,47.07,47.18,40.27,43.04,43.04,4674353
1,A,2000-01-04,40.72,41.17,38.70,39.75,39.75,4765083
2,A,2000-01-05,39.60,39.75,36.05,37.28,37.28,5758642
3,A,2000-01-06,36.83,37.06,34.74,35.86,35.86,2534434
4,A,2000-01-07,35.30,39.41,35.27,38.85,38.85,2819626


In [3]:
stock = df[df["Ticker"] == "AAPL"].copy()

stock["Date"] = pd.to_datetime(stock["Date"])
stock = stock.sort_values("Date")

stock.head()

,Ticker,Date,Open,High,Low,Close,Adj Close,Volume
11705,AAPL,2000-01-03,0.79,0.84,0.76,0.84,0.84,535796800
11706,AAPL,2000-01-04,0.81,0.83,0.76,0.77,0.77,512377600
11707,AAPL,2000-01-05,0.78,0.83,0.77,0.78,0.78,778321600
11708,AAPL,2000-01-06,0.79,0.80,0.71,0.71,0.71,767972800
11709,AAPL,2000-01-07,0.72,0.76,0.72,0.75,0.75,460734400


In [4]:
df.head()

,Ticker,Date,Open,High,Low,Close,Adj Close,Volume
0,A,2000-01-03,47.07,47.18,40.27,43.04,43.04,4674353
1,A,2000-01-04,40.72,41.17,38.70,39.75,39.75,4765083
2,A,2000-01-05,39.60,39.75,36.05,37.28,37.28,5758642
3,A,2000-01-06,36.83,37.06,34.74,35.86,35.86,2534434
4,A,2000-01-07,35.30,39.41,35.27,38.85,38.85,2819626


In [4]:
stock["Tomorrow_Close"] = stock["Close"].shift(-1)
stock["Target"] = (stock["Tomorrow_Close"] > stock["Close"]).astype(int)

stock = stock.dropna()

stock[["Date", "Close", "Tomorrow_Close", "Target"]].head()

,Date,Close,Tomorrow_Close,Target
11705,2000-01-03,0.84,0.77,0
11706,2000-01-04,0.77,0.78,1
11707,2000-01-05,0.78,0.71,0
11708,2000-01-06,0.71,0.75,1
11709,2000-01-07,0.75,0.73,0


In [3]:
df.columns

Index(['Ticker', 'Date', 'Open', 'High', 'Low', 'Close', 'Adj Close',
       'Volume'],
      dtype='object')

In [5]:
stock["Daily_Return"] = stock["Close"].pct_change()
stock["MA_5"] = stock["Close"].rolling(5).mean()
stock["MA_10"] = stock["Close"].rolling(10).mean()
stock["Volume_Change"] = stock["Volume"].pct_change()
stock["Volatility_5"] = stock["Daily_Return"].rolling(5).std()

stock = stock.dropna()

stock.head()

,Ticker,Date,Open,High,Low,Close,Adj Close,Volume,Tomorrow_Close,Target,Daily_Return,MA_5,MA_10,Volume_Change,Volatility_5
11714,AAPL,2000-01-14,0.75,0.77,0.74,0.75,0.75,390376000,0.78,1,0.041667,0.708,0.739,-0.621980,0.071394
11715,AAPL,2000-01-18,0.76,0.79,0.75,0.78,0.78,459177600,0.80,1,0.040000,0.718,0.733,0.176244,0.070929
11716,AAPL,2000-01-19,0.79,0.81,0.77,0.80,0.80,597643200,0.85,1,0.025641,0.740,0.736,0.301551,0.059204
11717,AAPL,2000-01-20,0.87,0.91,0.85,0.85,0.85,1831132800,0.83,0,0.062500,0.780,0.743,2.063923,0.032001
11718,AAPL,2000-01-21,0.86,0.86,0.83,0.83,0.83,495924800,0.80,0,-0.023529,0.802,0.755,-0.729170,0.032303


In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/SP500_Historical_Data.csv")
df.head()

,Ticker,Date,Open,High,Low,Close,Adj Close,Volume
0,A,2000-01-03,47.07,47.18,40.27,43.04,43.04,4674353
1,A,2000-01-04,40.72,41.17,38.70,39.75,39.75,4765083
2,A,2000-01-05,39.60,39.75,36.05,37.28,37.28,5758642
3,A,2000-01-06,36.83,37.06,34.74,35.86,35.86,2534434
4,A,2000-01-07,35.30,39.41,35.27,38.85,38.85,2819626
